# Resume Categorizer 

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
clean_df = pd.read_csv(r'clean_resume_data.csv')
print(clean_df.head())
jobs_dataset = pd.read_csv(r'jobs_dataset_with_features.csv')
print(jobs_dataset.head())

         ID Category                                            Feature
0  16852973       HR  hr administrator marketing associate hr admini...
1  22323967       HR  hr specialist hr operations summary media prof...
2  33176873       HR  hr director summary years experience recruitin...
3  27018550       HR  hr specialist summary dedicated driven dynamic...
4  17812897       HR  hr manager skill highlights hr skills hr depar...
                        Role  \
0       Social Media Manager   
1     Frontend Web Developer   
2    Quality Control Manager   
3  Wireless Network Engineer   
4         Conference Manager   

                                            Features  
0  5 to 15 Years Digital Marketing Specialist M.T...  
1  2 to 12 Years Web Developer BCA HTML, CSS, Jav...  
2  0 to 12 Years Operations Manager PhD Quality c...  
3  4 to 11 Years Network Engineer PhD Wireless ne...  
4  1 to 12 Years Event Manager MBA Event planning...  


In [3]:
clean_df['Category'].value_counts()

Category
INFORMATION-TECHNOLOGY    120
BUSINESS-DEVELOPMENT      120
ADVOCATE                  118
CHEF                      118
FINANCE                   118
ENGINEERING               118
ACCOUNTANT                118
FITNESS                   117
AVIATION                  117
SALES                     116
HEALTHCARE                115
CONSULTANT                115
BANKING                   115
CONSTRUCTION              112
PUBLIC-RELATIONS          111
HR                        110
DESIGNER                  107
ARTS                      103
TEACHER                   102
APPAREL                    97
DIGITAL-MEDIA              96
AGRICULTURE                63
AUTOMOBILE                 36
BPO                        22
Name: count, dtype: int64

# Balanced dataset

### In your original data, categories like "BPO" (22 samples) and "AUTOMOBILE" (36 samples) were significantly underrepresented compared to categories like "INFORMATION-TECHNOLOGY" (120 samples) Without balancing, your model would naturally perform better on majority classes while ignoring minority classes

In [7]:
!pip install imbalanced-learn

  Using cached imbalanced_learn-0.14.1-py3-none-any.whl.metadata (8.9 kB)
  Using cached sklearn_compat-0.1.5-py3-none-any.whl.metadata (20 kB)
Using cached imbalanced_learn-0.14.1-py3-none-any.whl (235 kB)
Using cached sklearn_compat-0.1.5-py3-none-any.whl (20 kB)

   ---------------------------------------- 0/2 [sklearn-compat]
   -------------------- ------------------- 1/2 [imbalanced-learn]
   -------------------- ------------------- 1/2 [imbalanced-learn]



ERROR: Could not install packages due to an OSError: [WinError 32] The process cannot access the file because it is being used by another process: 'D:\\PROJECTS\\ai_interview_analysis\\Ai_interview_Latest\\AI-Interview\\backend\\env\\Lib\\site-packages\\imblearn\\over_sampling\\_smote\\tests\\test_borderline_smote.py'
Check the permissions.


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [8]:
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler
from imblearn.pipeline import Pipeline

#Combine over- and under-sampling using SMOTE and Edited Nearest Neighbours.
from imblearn.combine import SMOTEENN

X_train, X_test, y_train, y_test = train_test_split(
    clean_df['Feature'], clean_df['Category'], test_size=0.2, random_state=42
)

pipeline = Pipeline([
    ('tfidf', TfidfVectorizer()),
    ('sampling', SMOTEENN(  
        sampling_strategy='auto',  
        random_state=42
    ))
])

# X_train_res, y_train_res = pipeline.fit_resample(X_train, y_train)

In [9]:
from sklearn.utils import resample
max_count = clean_df['Category'].value_counts().max()

balanced_data = []

for category in clean_df['Category'].unique():
        category_data = clean_df[clean_df['Category'] == category]
        if len(category_data) < max_count:
            balanced_category_data = resample(category_data, replace=True, n_samples=max_count, random_state=42)
        else:
            balanced_category_data = resample(category_data, replace=False, n_samples=max_count, random_state=42)
        balanced_data.append(balanced_category_data)

balanced_df = pd.concat(balanced_data)

balanced_df.dropna(inplace=True) 
balanced_df.isnull().sum()

ID          0
Category    0
Feature     0
dtype: int64

# Splitting

In [10]:

X = balanced_df['Feature']
y = balanced_df['Category']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Encoding

In [11]:
# TF-IDF (Term Frequency - Inverse Document Frequency) -> text to neumaric

vectorizer = TfidfVectorizer()
X_train_encoded = vectorizer.fit_transform(X_train)
X_test_encoded = vectorizer.transform(X_test)


In [12]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
y_train_encoded = le.fit_transform(y_train)
y_test_encoded = le.transform(y_test)


# Model Selection

In [13]:

from sklearn.ensemble import  GradientBoostingClassifier
from sklearn.metrics import accuracy_score, classification_report , f1_score

GBClassifier = GradientBoostingClassifier()
model = GBClassifier.fit(X_train_encoded , y_train_encoded)
y_pred = model.predict(X_test_encoded)

f1 = f1_score(y_test_encoded , y_pred , average='weighted')
accuracy = accuracy_score(y_test_encoded, y_pred)

print(f' Accuracy: {accuracy:.4f}')
print(f'F1 Score: {f1:.4f}')


 Accuracy: 0.8819
F1 Score: 0.8786


# Predictive Category

In [14]:
import re

url_pattern = re.compile(r'http\S*')  
rt_cc_pattern = re.compile(r'\b(RT|cc)\b')
hashtag_pattern = re.compile(r'#\S*')
mention_pattern = re.compile(r'@\S+')
special_chars_pattern = re.compile(r'[%s]' % re.escape("""!"#$%&'()*+,-./:;<=>?@[\]^_`{|}~"""))
non_ascii_pattern = re.compile(r'[^\x00-\x7f]')
extra_spaces_pattern = re.compile(r'\s+')

def cleanResume(txt):
    txt = txt.lower()
    txt = url_pattern.sub(' ', txt)
    txt = rt_cc_pattern.sub(' ', txt)
    txt = hashtag_pattern.sub(' ', txt)
    txt = mention_pattern.sub(' ', txt)
    txt = special_chars_pattern.sub(' ', txt)
    txt = non_ascii_pattern.sub(' ', txt)
    txt = extra_spaces_pattern.sub(' ', txt).strip()
    return txt
def predict_category(resume_text):
    resume_text = cleanResume(resume_text)
    resume_tfidf = vectorizer.transform([resume_text])
    return GBClassifier.predict(resume_tfidf)[0]


<>:7: SyntaxWarning: "\]" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\]"? A raw string is also an option.
<>:7: SyntaxWarning: "\]" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\]"? A raw string is also an option.
C:\Users\Anirban\AppData\Local\Temp\ipykernel_16524\336779421.py:7: SyntaxWarning: "\]" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\]"? A raw string is also an option.
  special_chars_pattern = re.compile(r'[%s]' % re.escape("""!"#$%&'()*+,-./:;<=>?@[\]^_`{|}~"""))


In [15]:
category_list = clean_df['Category'].dropna().unique().tolist()
print(category_list)


['HR', 'DESIGNER', 'INFORMATION-TECHNOLOGY', 'TEACHER', 'ADVOCATE', 'BUSINESS-DEVELOPMENT', 'HEALTHCARE', 'FITNESS', 'AGRICULTURE', 'BPO', 'SALES', 'CONSULTANT', 'DIGITAL-MEDIA', 'AUTOMOBILE', 'CHEF', 'FINANCE', 'APPAREL', 'ENGINEERING', 'ACCOUNTANT', 'CONSTRUCTION', 'PUBLIC-RELATIONS', 'BANKING', 'ARTS', 'AVIATION']


In [16]:

resume_file = """PRINCE RAJ
+91 9199322688 ⋄Kolkata, India
profile.princeraj@gmail.com ⋄linkedin.com/in/theprinceraj ⋄prince.is-a.dev
SKILLS
Languages JavaScript, TypeScript, HTML, CSS, C++, C
Technologies/Frameworks Node.js, NestJS, ExpressJS, Google Firebase, Vercel, RESTful API,
MongoDB, ElectronJS, Selenium Webdriver, Canvas
Developer Tools Visual Studio Code, Git, GitHub, Vercel, Postman, MongoDB Compass
Soft Skills Team Collaboration, Time Management, Problem-Solving, Attention to Detail
PROJECTS
Backend for Social Media Platform.
•Solely responsible for designing and building a robust and scalable backend API from scratch using NestJS for
a social media platform, encompassing 30+ endpoints for user management, post creation, social connections,
and other core features without compromising security and scalability.
•Implemented 5+ core features, like sign-in with Google (leveraging OAuth 2.0), OTP-based verification (using
Brevo), profile customization (including profile picture upload, bio, location, etc.), and hashtags support in
posts, etc.
•Designed and optimized the database schema using MongoDB, incorporating geospatial indexing for efficient
retrieval of location-based data, to ensure scalability and efficient handling of a large volume of user data and
concurrent requests.
Profile Cards Generator. (Try it here)
•Created a full-stack website that generates shareable and personalized profile cards based on user-input data in
real-time using a scalable API consisting of 5 endpoints, built using NodeJS and Express.
•Optimized API performance by implementing debouncing and field validations on front-end, resulting in signif-
icantly less load on server.
•Deployed on Vercel for efficient and reliable profile rendering, leveraging their platform’s high availability infras-
tructure and achieving a 99.95% uptime rate.
OCR-based Marksheet Scanner and CSV Converter Tool. (View here)
•Developed an open-source web-based OCR application using Tesseract.js that automates the conversion of
marksheets to CSV, handling 20+ fields of data per marksheet like, student name, subject, grades.
•Developed custom utility functions with the proper documentation to extract relevant data from the marksheets,
ensuring accuracy in the converted CSV format.
•Automated the marksheet conversion process, eliminating manual data entry, and significantly reducing the time
required to generate CSV files.
EDUCATION
Bachelor of Technology, Narula Institute of Technology 2023 - 2027
Computer Science Engineering With Specialization in Artificial Intelligence and Machine Learning
EXTRA-CURRICULAR ACTIVITIES
•Active member of the Coding Club and Google Developer Student Community at Narula Institute of Technology,
participating in workshops, hackathons, and events organized by Google Developer Groups, Kolkata.

"""
from sklearn.preprocessing import LabelEncoder
label_encoder = LabelEncoder()
# predicted_category = predict_category(resume_file)
# print("Predicted Category:", predicted_category)
# predicted_category = label_encoder.inverse_transform([predict_category(resume_file)])[0]
# print("Predicted Category:", predicted_category)
clean_df['Category_Encoded'] = label_encoder.fit_transform(clean_df['Category'])
predicted_category = label_encoder.inverse_transform([predict_category(resume_file)])[0]
print("Predicted Category:", predicted_category)



Predicted Category: INFORMATION-TECHNOLOGY


'\n    _Predicted Category: HR\n\n\n'

# Model Saving

In [18]:
import pickle
pickle.dump(GBClassifier,open('saved_models/gb_classifier_categorization.pkl','wb'))
pickle.dump(vectorizer,open('saved_models/tfidf_vectorizer_categorization.pkl','wb'))